<a href="https://colab.research.google.com/github/bilalsherifdeen1-ux/AI-Project-Gallery/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install duckdb datasets --quiet

from google.colab import userdata
import duckdb

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

con.sql(f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{userdata.get("HF_TOKEN")}'
    );
""")

MARCH_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
FEB_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet"
CONTENT_PATH = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"
CLIENTS_PATH = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"

print("Setup complete.")

Setup complete.


In [ ]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{MARCH_PATH}') LIMIT 1").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [ ]:
staleness_bucket_query = f"""
WITH march_imp AS (
SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_march
FROM read_parquet('{MARCH_PATH}')
GROUP BY client_hash_id, content_hash_id
),
feb_imp AS (
SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_feb
FROM read_parquet('{FEB_PATH}')
GROUP BY client_hash_id, content_hash_id
),
joined AS (
SELECT m.client_hash_id, m.content_hash_id,
m.impressions_march, f.impressions_feb,
(m.impressions_march < f.impressions_feb) AS is_declining,
d.content_updated_date,
DATE_DIFF('month', d.content_updated_date, DATE '2026-03-31') AS months_since_update
FROM march_imp m
JOIN feb_imp f ON m.client_hash_id = f.client_hash_id AND m.content_hash_id = f.content_hash_id
JOIN read_parquet('{CONTENT_PATH}') d ON m.content_hash_id = d.content_hash_id
WHERE f.impressions_feb >= 50
)
SELECT
CASE
WHEN months_since_update <= 3 THEN '0-3 months'
WHEN months_since_update <= 6 THEN '3-6 months'
WHEN months_since_update <= 12 THEN '6-12 months'
ELSE '12+ months'
END AS staleness_band,
ROUND(AVG(CASE WHEN is_declining THEN 1.0 ELSE 0.0 END) * 100, 1) AS pct_declining,
COUNT(*) AS n
FROM joined
GROUP BY staleness_band
ORDER BY MIN(months_since_update)
"""
con.sql(staleness_bucket_query).show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬───────────────┬───────┐
│ staleness_band │ pct_declining │   n   │
│    varchar     │    double     │ int64 │
├────────────────┼───────────────┼───────┤
│ 0-3 months     │          31.1 │ 88985 │
│ 3-6 months     │          17.4 │    69 │
│ 6-12 months    │          71.9 │    32 │
└────────────────┴───────────────┴───────┘



In [ ]:
import pandas as pd

# Expected CTR by position band, from your own bucket table (Part 1, Signal 1)
expected_ctr_by_band = {
    "1-3": 0.34, "4-6": 0.34, "7-10": 0.29, "11-20": 0.24, "21+": 0.13
}

scoring_query = f"""
SELECT client_hash_id, content_hash_id,
SUM(gsc_clicks) AS clicks_march,
SUM(gsc_impressions) AS impressions_march,
SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS avg_position_march
FROM read_parquet('{MARCH_PATH}')
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) >= 100
"""
scored = con.sql(scoring_query).df()

scored["actual_ctr_pct"] = (scored["clicks_march"] / scored["impressions_march"]) * 100

def position_band(pos):
    if pos <= 3: return "1-3"
    elif pos <= 6: return "4-6"
    elif pos <= 10: return "7-10"
    elif pos <= 20: return "11-20"
    else: return "21+"

scored["position_band"] = scored["avg_position_march"].apply(position_band)
scored["expected_ctr_pct"] = scored["position_band"].map(expected_ctr_by_band)

# THE RULE: score = how far actual CTR falls below what the position band predicts
scored["score"] = scored["expected_ctr_pct"] - scored["actual_ctr_pct"]

scored["reason_code"] = scored["score"].apply(
    lambda s: "CTR_BELOW_POSITION_PEERS" if s > 0.05 else "PERFORMING_AS_EXPECTED"
)
scored["action_label"] = scored["reason_code"].apply(
    lambda r: "review_snippet_title" if r == "CTR_BELOW_POSITION_PEERS" else "no_action"
)

ranked_queue = scored.sort_values("score", ascending=False)
ranked_queue.to_csv("baseline_action_score.csv", index=False)
print(ranked_queue.head(10)[["content_hash_id", "position_band", "expected_ctr_pct", "actual_ctr_pct", "score", "reason_code", "action_label"]])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                 content_hash_id position_band  expected_ctr_pct  \
1       content_a3ea9792f793ec72           4-6              0.34   
101428  content_a71d7608ea2c1456           4-6              0.34   
16738   content_3a80a9cabba56cf2           4-6              0.34   
16739   content_b0acd237c0d3482a           1-3              0.34   
16741   content_c61db8a6018a5010           1-3              0.34   
16753   content_4b70641bfa5308dc           4-6              0.34   
16759   content_7da0f247b765dffe           1-3              0.34   
16763   content_3614336490a559ab           1-3              0.34   
16788   content_738f3a26084f61b2           1-3              0.34   
16791   content_f1bae964f19289fa           4-6              0.34   

        actual_ctr_pct  score               reason_code          action_label  
1                  0.0   0.34  CTR_BELOW_POSITION_PEERS  review_snippet_title  
101428             0.0   0.34  CTR_BELOW_POSITION_PEERS  review_snippet_title  
16738      

In [ ]:
print((scored["actual_ctr_pct"] == 0).sum(), "of", len(scored), "items have zero clicks")

37779 of 101441 items have zero clicks


In [ ]:
ranked_queue = scored.sort_values(["score", "impressions_march"], ascending=[False, False])

In [ ]:
ranked_queue.to_csv("baseline_action_score.csv", index=False)
print(ranked_queue.head(10)[["content_hash_id", "position_band", "actual_ctr_pct", "impressions_march", "score", "reason_code", "action_label"]])

                content_hash_id position_band  actual_ctr_pct  \
26256  content_bf078007df823490           1-3             0.0   
57495  content_0c5606abaaab3178           4-6             0.0   
56824  content_713b157e9c77690a           4-6             0.0   
50816  content_c9f840183215651b           4-6             0.0   
14573  content_fe8baba849843607           4-6             0.0   
82488  content_fa17add7836d36c3           1-3             0.0   
35381  content_b154f6c2652cfeb9           1-3             0.0   
54932  content_0e2e4d3ab02abc1a           4-6             0.0   
19821  content_f5a7a2559d483a54           1-3             0.0   
82941  content_520e203a08cd69ee           1-3             0.0   

       impressions_march  score               reason_code  \
26256            44707.0   0.34  CTR_BELOW_POSITION_PEERS   
57495            38865.0   0.34  CTR_BELOW_POSITION_PEERS   
56824            24908.0   0.34  CTR_BELOW_POSITION_PEERS   
50816            21519.0   0.34  CTR_BEL

In [ ]:
import os
os.makedirs("work/outputs", exist_ok=True)
ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

In [ ]:
!ls -la work/outputs/

ls: cannot access 'work/outputs/': No such file or directory


In [ ]:
!pip install duckdb datasets scikit-learn --quiet

from google.colab import userdata
import duckdb

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{userdata.get("HF_TOKEN")}'
    );
""")

MARCH_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
CONTENT_PATH = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

print("Setup complete.")

Setup complete.


In [ ]:
feature_query = f"""
WITH march_agg AS (
SELECT
client_hash_id, content_hash_id,
SUM(gsc_clicks) AS clicks_march,
SUM(gsc_impressions) AS impressions_march,
SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS avg_position_march
FROM read_parquet('{MARCH_PATH}')
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) >= 100
)
SELECT
m.client_hash_id, m.content_hash_id,
m.avg_position_march, m.impressions_march,
d.main_intent, d.search_volume, d.category_count,
m.clicks_march * 1.0 / NULLIF(m.impressions_march, 0) AS actual_ctr_march
FROM march_agg m
JOIN read_parquet('{CONTENT_PATH}') d ON m.content_hash_id = d.content_hash_id
"""
df = con.sql(feature_query).df().dropna(subset=["actual_ctr_march", "avg_position_march", "main_intent",
                                                 "search_volume", "category_count", "impressions_march"])

numeric_cols_to_fix = ["avg_position_march", "impressions_march", "search_volume", "category_count"]
df[numeric_cols_to_fix] = df[numeric_cols_to_fix].astype("float64")

print(df.shape)
df.dtypes

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(99197, 8)


,0
client_hash_id,object
content_hash_id,object
avg_position_march,float64
impressions_march,float64
main_intent,object
search_volume,float64
category_count,float64
actual_ctr_march,float64


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

numeric_features = ["avg_position_march", "impressions_march", "search_volume", "category_count"]
categorical_features = ["main_intent"]

def build_pipeline():
    pre = ColumnTransformer([
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ])
    return Pipeline([
        ("pre", pre),
        ("rf", RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)),
    ])

X = df[numeric_features + categorical_features]
y = df["actual_ctr_march"]
groups = df["client_hash_id"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.25, random_state=42)
model_random = build_pipeline()
model_random.fit(X_train_r, y_train_r)
mae_random_split = mean_absolute_error(y_test_r, model_random.predict(X_test_r))

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]
model_grouped = build_pipeline()
model_grouped.fit(X_train_g, y_train_g)
mae_grouped_split = mean_absolute_error(y_test_g, model_grouped.predict(X_test_g))

print(f"MAE, naive random split:      {mae_random_split:.5f}")
print(f"MAE, grouped-by-client split: {mae_grouped_split:.5f}")
print(f"Gap: {mae_grouped_split - mae_random_split:.5f}")

overlap = X_test_g.index.isin(X_train_g.index).sum()
print(f"\nSanity check — rows overlapping between grouped train/test: {overlap} (should be 0)")

MAE, naive random split:      0.00245
MAE, grouped-by-client split: 0.00268
Gap: 0.00023

Sanity check — rows overlapping between grouped train/test: 0 (should be 0)


In [ ]:
expected_ctr_by_band = {
    "1-3": 0.0034, "4-6": 0.0034, "7-10": 0.0029, "11-20": 0.0024, "21+": 0.0013
}

def position_band(pos):
    if pos <= 3: return "1-3"
    elif pos <= 6: return "4-6"
    elif pos <= 10: return "7-10"
    elif pos <= 20: return "11-20"
    else: return "21+"

X_test_g = X_test_g.copy()
X_test_g["position_band"] = X_test_g["avg_position_march"].apply(position_band)
baseline_preds = X_test_g["position_band"].map(expected_ctr_by_band)

mae_baseline = mean_absolute_error(y_test_g, baseline_preds)
mae_model = mae_grouped_split

print(f"Week 4 baseline MAE: {mae_baseline:.5f}")
print(f"Random Forest MAE:   {mae_model:.5f}")

if mae_model < mae_baseline:
    print(f"\nBeats baseline by {(1 - mae_model / mae_baseline) * 100:.1f}%")
else:
    print(f"\nDoes NOT beat baseline — {(mae_model / mae_baseline - 1) * 100:.1f}% worse")

Week 4 baseline MAE: 0.00277
Random Forest MAE:   0.00268

Beats baseline by 3.2%


In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(model_grouped, X_test_g[numeric_features + categorical_features],
                               y_test_g, n_repeats=10, random_state=42, n_jobs=-1)

importance_df = pd.DataFrame({
    "feature": numeric_features + categorical_features,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

importance_df

,feature,importance_mean,importance_std
0,avg_position_march,0.079851,0.002823
2,search_volume,0.010936,0.001002
3,category_count,0.001578,0.001191
4,main_intent,-0.000023,0.000193
1,impressions_march,-0.001345,0.000788


In [ ]:
X_test_g["actual_ctr"] = y_test_g.values
X_test_g["predicted_ctr"] = model_grouped.predict(X_test_g[numeric_features + categorical_features])
X_test_g["abs_error"] = (X_test_g["actual_ctr"] - X_test_g["predicted_ctr"]).abs()

worst_errors = X_test_g.sort_values("abs_error", ascending=False).head(10)
worst_errors[["position_band", "impressions_march", "main_intent", "actual_ctr", "predicted_ctr", "abs_error"]]

,position_band,impressions_march,main_intent,actual_ctr,predicted_ctr,abs_error
39976,1-3,140.0,informational,0.078571,0.002846,0.075726
87397,21+,160.0,informational,0.075000,0.001423,0.073577
50797,11-20,259.0,informational,0.073359,0.002015,0.071344
90765,4-6,111.0,transactional,0.072072,0.003117,0.068955
39884,1-3,181.0,informational,0.066298,0.002587,0.063711
101138,4-6,307.0,transactional,0.065147,0.002981,0.062166
36597,21+,123.0,informational,0.056911,0.001649,0.055261
101132,4-6,164.0,transactional,0.054878,0.002884,0.051994
39777,4-6,108.0,informational,0.055556,0.004177,0.051378
50788,21+,124.0,informational,0.048387,0.000896,0.047492
